In [ ]:
import source.data
import source.predict
import source.helpers
import biolib
from pathlib import Path
import pandas as pd

from source import regions, analysis
from source import alphamissense as am

In [ ]:
# some libraries that might be neeeded
#pip install pymissense
#pip3 install --upgrade pybiolib 
#pip install gunicorn

## get data

In [ ]:
source.data.import_alphamissense()

In [ ]:
RES_DIR = Path.cwd() / 'results'
DATA_DIR = Path.cwd() / 'data'

In [ ]:
proteins = {
'GLUT1':'P11166',
'GLUT2':'P11168',
'GLUT3':'P11169',
#'GLUT4':'P14672',
#'GLUT5':'P22732',
#'GLUT6':'Q9UGQ3',
#'GLUT7':'Q6PXP3',
#'GLUT8':'Q9NY64',
#'GLUT9':'Q9NRM0',
#'GLUT10':'O95528',
#'GLUT11':'Q9BYW1',
#'GLUT12':'Q8TD20',
#'GLUT13':'Q96QE2',
#'GLUT14':'Q8TDB8' 
}

In [ ]:
#import pdb and fasta files
for up_id in proteins.values():
    source.data.import_pdb(up_id)
    source.data.import_fasta(up_id)

## get predictions for proteins

In [ ]:
# run pymissense for all proteins and generate csv of aminoacid pathogenicities
for up_id in proteins.values():
    print(f'running PyMissense for {up_id}')
    source.predict.pymissense(up_id)

## get protein regions
### all

In [ ]:
REG_DIR = RES_DIR / 'regions'

In [ ]:
for up_id in proteins.values():
    regions.all_residues(pdb_file = DATA_DIR / f'pdb/{up_id}.pdb', out_file = REG_DIR/ f'{up_id}.csv')

### intracellular, extracellular and membrane regions

In [ ]:
#get annotations of AA positions relative to the membrane using 
deeptmhmm = biolib.load('DTU/DeepTMHMM')
for up_id in proteins.values():
    source.predict.deepTMHMM(up_id,deeptmhmm)

In [ ]:
#processing of depptmhmm results
for up_id in proteins.values():
    regions.membrane_residues(deeptmhmm_file = RES_DIR / f'deeptmhmm/{up_id}.3line', out_dir = REG_DIR, identifier = up_id)

### binding pockets, lining residues and nonpocket binding residues

In [ ]:
# The amino acid residues lining the pores of individual proteins were calculated using: https://mole.upol.cz/ -LINING RESIDUES
# The amino acid residues framing the protein binding site were calculated using: https://prankweb.cz/ - BINDING PLACE
# The results were transcribed into an .xlsx document. Saved here /GLUT project documentation/Binding places and lining residues/Excel files/
# After transcription, a third column was created, where only those amino acid residues that were not part of the binding site but framed the protein pore were transcribed. -BINDING PLACE-LINING RESIDUES

In [ ]:
#this step is hard to automatize
#uses csv file from PrankWeb
for up_id in proteins.values():
    regions.binding_pockets(prankweb_csv = DATA_DIR / f'prankweb/{up_id}.csv', out_file = REG_DIR/ f'bp_{up_id}.csv')

In [ ]:
##unziper na mole online
#for glut in proteins.keys():
#    file_path = DATA_DIR / f'moleonline/{glut}.zip'
#    output = DATA_DIR / f'moleonline/{proteins[glut]}.json'
#    source.helpers.mole_zip2json(file_path, output)

In [ ]:
#lining residues
#from manually obrained JSON files from mole online
for up_id in proteins.values():
    regions.lining_residues(moleonline_json = DATA_DIR / f'moleonline/{up_id}.json', out_file = REG_DIR/ f'lr_{up_id}.csv')

In [ ]:
#nonbinding pocket lining residues
for up_id in proteins.values():
    regions.nonbinding_pocket_lining_residues(bp_residues_csv = REG_DIR / f'bp_{up_id}.csv', l_residues_csv = REG_DIR / f'lr_{up_id}.csv', out_file = REG_DIR / f'nobp_lr_{up_id}.csv')

## assign pathogenicites

In [ ]:
AM_DIR = RES_DIR / 'pathogenicities/alphamissense'

In [ ]:
for up_id in proteins.values():
    patho_csv = AM_DIR / f'{up_id}.csv'
    #create file for
    am.residues(RES_DIR / f'pymissense/{up_id}-edit.pdb', patho_csv)
    #intracellular
    analysis.assign_pathogenicity(patho_csv, region_csv = REG_DIR / f'I_{up_id}.csv',out_file = AM_DIR / f'I_{up_id}.csv')

## average pathogenicities


In [ ]:
#whole proteins
analysis.average_pathogenicity(patho_dir = AM_DIR, region_prefix = '', up_id_list = proteins.values(), out_file = AM_DIR / 'all_average_patho.csv')
#intracellular amino acids
analysis.average_pathogenicity(patho_dir = AM_DIR, region_prefix = 'I_', up_id_list = proteins.values(), out_file = AM_DIR / 'I_average_patho.csv')